## Prepare data set from survey data for the choice model
Based on Graham (2020) and Joon (2025, https://github.com/krisjuune/cantonal-preferences-s2z/)

### Load libraries

In [1]:
import pandas as pd
import numpy as np
import os
import re
import importlib
import f_data_processing as dp
import map_data as md

dp = importlib.reload(dp)
md = importlib.reload(md)
print(os.getcwd())


c:\Users\stdt\ETH Zurich\PhD Veronika - 07_Master Thesis Natalie\10_hcm_code\natural_hazard_solidarity


### Read surveys and do first filtering

Define the data file paths to the survey-data as dictionary. In the dictionary you can always add more datasets to merge (i.e. after another survey round S2).


In [2]:
survey_dict = {'S0': '../data/Survey_Adaptation Natural Hazards_First Wave_raw data.csv', 
               'S1': '../data/Survey_Adaptation Natural Hazards_Second Wave_raw data.csv'}

Loop through the dictionary and apply filtering and cleansing and type conversion. The following steps are done:

1. Read csv-file and clean column names for trailing spaces, dots, and naming as well as change cell values to numeric for the necessary id-columns.
2. Remove entries that are not complete.
3. Filter out speedies and slowies, and inattentives.
4. Transform needed likert/numeric columns (defined in `map_data.py`) into numeric columns.
5. Transform needed non-numeric cells to numeric value.
6. Translate choice experiment replies (choice descriptions) to english.
7. Drop additional responses where more than two have been done using the same IPA-address.
8. Delete unecessary or unanonymos columns.
9. Add prefix to every column name so the different survey can be merged.


In [ ]:
df_dict ={}

# language region is kept to analyze eventual cultural differences within Switzerland
#p5, p95 = 0

for s, file in survey_dict.items():
    df = pd.read_csv(file, dtype={'id': 'string'}, skiprows=[1,2])
    # remove lines, replace empty cells, rename columnames
    df = df.replace('', pd.NA)
    df.columns = df.columns.str.replace('.', '', regex=False)
    df.columns = df.columns.str.replace(r'\s+', '', regex=True)
    df = dp.rename_columns(df, 'municipality', 'benefits')
    df = dp.to_num_col(df, ['id', 'm'])
    print("original länge: ",len(df))

    # filter only complete/valid response rows
    df = df[(df['DistributionChannel'] != 'preview') & (df['Finished'] != False)].copy()
    df = df[~df['Q_TerminateFlag'].isin(['PoorQuality', 'NA', 'QuotaMet', 'Screened'])].copy()

    # Exclude speeders, slowers, straightliners, inattentives
    speedy_slowy = dp.rm_speeders_slowers(df,s, speedy=300, slowy=60*60*24)
    
    # Exclude straightliners, inattentives
    # straightliners = dp.rm_straightliners(df)
    # inattentives = df[f'attention_check'] != 'Agree'
    # df_filtered = df[~((speedy_slowy | straightliners | inattentives))]
    df_filtered = df[~((speedy_slowy))]
    df_filtered = df_filtered.copy()
    
    # transform liker/numeric scales, non-numeric columns
    if s == 'S0':
        df_filtered[md.NUM_COLUMNS] = df_filtered[md.NUM_COLUMNS].apply(pd.to_numeric, errors="coerce")
    df_filtered = dp.string_mapping(df_filtered, md.LIKERT_MAP, md.VALID_COLUMNS, numeric=True)
    df_filtered = dp.string_mapping(df_filtered, md.LIKERT_MAP, ['climatechange_nh_1'], numeric=True)
    df_filtered = dp.string_mapping(df_filtered, md.NH_EXPERIENCE_MAP,['experience_nh'],numeric=True)
    
    # invert likert scales for consistency
    dp.invert_likert(df_filtered, ['finan_vulnerability_1'], 6)
    
    # remove double ipas
    df_filtered = dp.filter_double_ipa(df_filtered)
    
    pattern = '|'.join(map(re.escape, md.ANONYMIZE_COLS))
    columns = df_filtered.filter(regex=pattern)
    df_filtered = df_filtered.drop(columns=df_filtered.filter(regex=pattern).columns)
    
    # prepare for merging
    df_filtered = df_filtered.add_prefix(f'{s}_')
    df_dict[s] = df_filtered
    df_filtered.to_csv(f'../results/{s}_clean_survey.csv')
    print(len(df_filtered))
    
df_filtered



original länge:  1528
Fastest 5% and slowest 5% (Total): 35 respondents (<300s or >86400s)
1125
original länge:  774
Fastest 5% and and all slowest (Total): 2 respondents (<300s or >86400s)
592


,S1_StartDate,S1_EndDate,S1_Progress,S1_duration,S1_Finished,S1_language,S1_knowledge_nh_1,S1_knowledge_nh_2,S1_knowledge_nh_3,S1_sensitivity_nh_1,...,S1_choice9_costs1,S1_choice9_costs2,S1_choice9_exemptions1,S1_choice9_exemptions2,S1_choice9_benefits1,S1_choice9_benefits2,S1_Q_TerminateFlag,S1_Q_R_Del,S1_screened_out,S1_SelectedLanguage
0,2025-08-30 16:55:44,2025-08-30 17:06:19,100,635,True,Français,Yes,Yes,Yes,1.0,...,Tous les citoyens paient le même montant,Les personnes paient proportionnellement à leu...,Les personnes à faibles et moyens revenus peuv...,Les personnes à faibles et moyens revenus peuv...,Municipalités ayant une grande valeur culturel...,Niveaux de protection égaux pour toutes les mu...,Complete,NaN,False,FR
1,2025-08-30 17:00:48,2025-08-30 17:07:25,100,397,True,Français,Yes,Yes,Yes,6.0,...,Les personnes et entreprises bénéficiant des m...,Les personnes et entreprises bénéficiant des m...,Les personnes à faibles et moyens revenus peuv...,Les personnes à faibles et moyens revenus peuv...,Niveaux de protection égaux pour toutes les mu...,Niveaux de protection égaux pour toutes les mu...,Complete,NaN,False,FR
2,2025-08-30 16:55:45,2025-08-30 17:09:24,100,818,True,Français,Yes,No,Yes,6.0,...,Les entreprises paient proportionnellement à l...,Tous les citoyens paient le même montant,Aucun groupe n'est exempté des coûts,Aucun groupe n'est exempté des coûts,Les municipalités les plus touchées par les ri...,Niveaux de protection égaux pour toutes les mu...,Complete,NaN,False,FR
3,2025-08-30 16:55:27,2025-08-30 17:09:25,100,837,True,Français,Yes,Yes,Yes,2.0,...,Tous les citoyens paient le même montant,Les entreprises paient proportionnellement à l...,Les personnes à faible revenu peuvent être exe...,Aucun groupe n'est exempté des coûts,Municipalités ayant une grande valeur culturel...,Les municipalités économiquement prospères,Complete,NaN,False,FR
4,2025-08-30 16:55:32,2025-08-30 17:11:05,100,933,True,Français,Yes,Yes,Yes,3.0,...,Les personnes paient proportionnellement à leu...,Les personnes paient proportionnellement à leu...,Les personnes à faibles et moyens revenus peuv...,Les personnes à faibles et moyens revenus peuv...,Les municipalités économiquement prospères,Niveaux de protection égaux pour toutes les mu...,Complete,NaN,False,FR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
587,2025-09-12 12:49:21,2025-09-12 13:09:29,100,1208,True,Deutsch,Yes,Yes,Yes,5.0,...,Unternehmen zahlen proportional zu ihrem CO2-A...,Alle Menschen zahlen den gleichen Betrag,Keine Gruppen sind von den Kosten ausgenommen,mit Ausnahme von Menschen mit niedrigem Einkommen,"Gemeinden, in denen Menschen seit vielen Jahre...",Wirtschaftlich wohlhabende Gemeinden,Complete,NaN,False,DE
588,2025-09-12 13:06:28,2025-09-12 13:29:18,100,1369,True,Deutsch,Yes,Yes,Yes,1.0,...,Menschen zahlen proportional zu ihrem CO2-Auss...,Alle Menschen zahlen den gleichen Betrag,mit Ausnahme von Menschen mit niedrigem und mi...,Keine Gruppen sind von den Kosten ausgenommen,Gleiche Schutzniveaus für alle Gemeinden,Wirtschaftlich wohlhabende Gemeinden,Complete,NaN,False,DE
589,2025-09-12 18:23:30,2025-09-12 18:36:10,100,759,True,Deutsch,Yes,Yes,Yes,4.0,...,Menschen zahlen proportional zu ihrem CO2-Auss...,Menschen zahlen proportional zu ihrem Einkommen,Keine Gruppen sind von den Kosten ausgenommen,mit Ausnahme von Menschen mit niedrigem Einkommen,Gemeinden mit vielen Kulturgütern wie z.B. his...,"Gemeinden, in denen Menschen seit vielen Jahre...",Complete,NaN,False,DE
590,2025-09-13 11:21:48,2025-09-13 11:34:14,100,746,True,Français,Yes,Yes,Yes,2.0,...,Les personnes paient proportionnellement à leu...,Les personnes et entreprises bénéficiant des m...,Aucun groupe n'est exempté des coûts,Aucun groupe n'est exempté des coûts,Municipalités ayant une grande valeur culturel...,Les municipalités économiquement prospères,Complete,NaN,False,FR


### Merging and cleanup
Merge the datasets on the user id so only respondents are kept that filled out both survey. This makes it possible to analyze the change in preference related to a natural hazard occurence (Blatten).
Afterwards the remaining answers are all standardized by translating answers to English.

In [ ]:
# merge both surveys and check for uniqueness
# here you can always add more dataframes to merge (i.e. after another survey round S2)
merged_waves_df = (
    md.KEYS
    .merge(df_dict['S0'], how='inner', left_on='S0_idx', right_on='S0_id')
    .merge(df_dict['S1'], how='inner', left_on='S1_idx', right_on='S1_m')
    .drop(columns=['S0_id', 'S0_m', 'S1_id', 'S1_m'])
)

print(len(merged_waves_df))

# drop non-unique rows and reset unique ids
df_cleaned = merged_waves_df.drop_duplicates(subset=['S0_idx', 'S1_idx'], keep='first')
df_cleaned = df_cleaned.reset_index(drop=True)
df_cleaned['respondent_id'] = df_cleaned.index + 1

# map demographics and translate choice options
df_cleaned = dp.string_mapping(df_cleaned, md.DEMOGRAPHICS_DICT, column_patterns=[r'_gender$', r'_age$', r'_education$', r'_income$', r'_language$', r'_language_region$', r'_party_choice$'])
df_cleaned = dp.string_mapping(df_cleaned, md.TRANSLATION_DICT, column_patterns=[r'^S._choice._exemptions', r'^S._choice._costs', r'^S._choice._benefits'])


587


Index(['S0_choice1_costs1', 'S0_choice1_costs2', 'S0_choice1_exemptions1',
       'S0_choice1_exemptions2', 'S0_choice1_benefits1',
       'S0_choice1_benefits2', 'S0_choice2_costs1', 'S0_choice2_costs2',
       'S0_choice2_exemptions1', 'S0_choice2_exemptions2',
       'S0_choice2_benefits1', 'S0_choice2_benefits2', 'S0_choice3_costs1',
       'S0_choice3_costs2', 'S0_choice3_exemptions1', 'S0_choice3_exemptions2',
       'S0_choice3_benefits1', 'S0_choice3_benefits2', 'S0_choice4_costs1',
       'S0_choice4_costs2', 'S0_choice4_exemptions1', 'S0_choice4_exemptions2',
       'S0_choice4_benefits1', 'S0_choice4_benefits2', 'S0_choice5_costs1',
       'S0_choice5_costs2', 'S0_choice5_exemptions1', 'S0_choice5_exemptions2',
       'S0_choice5_benefits1', 'S0_choice5_benefits2', 'S0_choice6_costs1',
       'S0_choice6_costs2', 'S0_choice6_exemptions1', 'S0_choice6_exemptions2',
       'S0_choice6_benefits1', 'S0_choice6_benefits2', 'S1_choice1_costs1',
       'S1_choice1_costs2', 'S1_choi

In [ ]:
# drop all entries that are not conjoint complete
choice_cols = df_cleaned.columns[df_cleaned.columns.str.contains(r"_choice\d+", case=False, na=False, regex=True)]

# see them
print(choice_cols.tolist())

# drop rows where ANY of those columns is NaN
df_clean = df_cleaned.dropna(subset=choice_cols)
df_clean

['S0_choice1_costs1', 'S0_choice1_costs2', 'S0_choice1_exemptions1', 'S0_choice1_exemptions2', 'S0_choice1_benefits1', 'S0_choice1_benefits2', 'S0_choice2_costs1', 'S0_choice2_costs2', 'S0_choice2_exemptions1', 'S0_choice2_exemptions2', 'S0_choice2_benefits1', 'S0_choice2_benefits2', 'S0_choice3_costs1', 'S0_choice3_costs2', 'S0_choice3_exemptions1', 'S0_choice3_exemptions2', 'S0_choice3_benefits1', 'S0_choice3_benefits2', 'S0_choice4_costs1', 'S0_choice4_costs2', 'S0_choice4_exemptions1', 'S0_choice4_exemptions2', 'S0_choice4_benefits1', 'S0_choice4_benefits2', 'S0_choice5_costs1', 'S0_choice5_costs2', 'S0_choice5_exemptions1', 'S0_choice5_exemptions2', 'S0_choice5_benefits1', 'S0_choice5_benefits2', 'S0_choice6_costs1', 'S0_choice6_costs2', 'S0_choice6_exemptions1', 'S0_choice6_exemptions2', 'S0_choice6_benefits1', 'S0_choice6_benefits2', 'S1_choice1_costs1', 'S1_choice1_costs2', 'S1_choice1_exemptions1', 'S1_choice1_exemptions2', 'S1_choice1_benefits1', 'S1_choice1_benefits2', 'S1_c

In [14]:
df_clean.to_csv("../results/full_merge_survey_conjoint.csv'")